# 03 - Remote Hugging Face Inference API Embeddings

> Complete guide and in-depth Arabic documentation: [`README.md`](./README.md)

Implements `HuggingFaceAPIEmbeddings` with remote serverless inference and FAISS integration.

## 1. Initialize Remote HuggingFaceAPIEmbeddings
Loads `HF_TOKEN` from `.env` and connects to the official remote Inference API.

In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# تحميل المتغيرات من .env في جذر المشروع
env_path = Path.cwd().parent.parent / ".env"
load_dotenv(env_path if env_path.exists() else Path("../../.env"))

# استيراد فئة التضمينات السحابية من 02-indexing
indexing_dir = Path.cwd().parent
if str(indexing_dir) not in sys.path:
    sys.path.insert(0, str(indexing_dir))

from huggingface_api import HuggingFaceAPIEmbeddings

print("⏳ جارٍ تهيئة HuggingFaceAPIEmbeddings (Remote Inference API)...")
embeddings = HuggingFaceAPIEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    normalize_embeddings=True,
    expected_dimension=384
)
print(f"✅ تم تهيئة كائن التضمين بنجاح: {embeddings}")

## 2. Embed Single Query with embed_query()

In [ ]:
sample_query = "Hello I am learning about embeddings"

query_vector = embeddings.embed_query(sample_query)

print(f"📝 نص الاستعلام: '{sample_query}'")
print(f"📏 أبعاد المتجه المسترجع من السيرفر: {len(query_vector)}")
print(f"🔍 عينة من أول 5 قيم: {query_vector[:5]}")
print(f"📌 نوع المتجه: {type(query_vector)} (1D list of floats)")

## 3. Embed Document Batch with embed_documents()

In [ ]:
# قائمة الجمل التجريبية من المحاضرة
sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",
    "The dog played in the yard",
    "I love programming in Python",
    "Python is my favorite programming language"
]

# استخراج مصفوفة المتجهات للدفعة الكاملة
doc_vectors = embeddings.embed_documents(sentences)

print(f"📚 عدد الجمل المسترجعة: {len(doc_vectors)}")
print(f"📏 أبعاد كل متجه: {len(doc_vectors[0])} بُعداً\n")

for idx, (sentence, vec) in enumerate(zip(sentences, doc_vectors), 1):
    print(f"[{idx}] '{sentence:45}' -> Vector sample: {vec[:3]}...")

## 4. Verify Determinism & L2 Normalization

In [ ]:
import numpy as np

vec_test_1 = embeddings.embed_query("The cat sat on the mat")
vec_test_2 = embeddings.embed_query("The cat sat on the mat")

is_identical = np.allclose(vec_test_1, vec_test_2)
norm_val = np.linalg.norm(vec_test_1)
print(f"🔒 هل التضمين متطابق وحتمي 100%؟ {is_identical}")
print(f"📏 معيار المتجه (L2 Norm): {norm_val:.6f} (معاير بنجاح)")

## 5. Pairwise Semantic Similarity Matrix

In [ ]:
import pandas as pd

mat = np.array(doc_vectors)
# بما أن المتجهات معيرة (Normalized)، فإن الضرب النقطي يمثل تشابه جيب التمام مباشرة
sim_matrix = np.dot(mat, mat.T)

short_labels = ["[1] Cat-Mat", "[2] Feline-Rug", "[3] Dog-Yard", "[4] Love-Python", "[5] Favorite-Lang"]
df_sim = pd.DataFrame(sim_matrix, index=short_labels, columns=short_labels)

print("📊 مصفوفة تشابه جيب التمام الدلالي بين الجمل (Cosine Similarity Matrix):")
display(df_sim.round(4))

print("\n🔍 نتائج التحليل الدلالي:")
print(f"✨ تشابه القطط [1] و [2]: {df_sim.iloc[0, 1]:.4f} (معاني متطابقة بمفردات مختلفة!)")
print(f"✨ تشابه بايثون [4] و [5]: {df_sim.iloc[3, 4]:.4f} (شغف البرمجة المشترك)")
print(f"❌ تشابه القطط مع بايثون [1] و [4]: {df_sim.iloc[0, 3]:.4f} (انعدام الصلة الدلالية تقريباً)")

## 6. End-to-End FAISS Vector Store Integration
Indexes documents in FAISS and performs semantic search via retriever.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# إنشاء وثائق LangChain تجريبية
documents = [
    Document(page_content="LangChain is a popular framework for developing LLM and RAG applications.", metadata={"source": "doc1", "topic": "ai"}),
    Document(page_content="Cats and kittens are small carnivorous mammals kept as indoor pets.", metadata={"source": "doc2", "topic": "animals"}),
    Document(page_content="Python is an interpreted high-level general-purpose programming language.", metadata={"source": "doc3", "topic": "tech"})
]

# 1. إنشاء فهرس FAISS وفهرسة الوثائق باستخدام التضمينات السحابية
print("⏳ جارٍ إنشاء فهرس FAISS وفهرسة المستندات...")
vectorstore = FAISS.from_documents(documents, embeddings)
print("✅ تم إنشاء فهرس FAISS بنجاح!")

# 2. تحويل الفهرس إلى مسترجع (Retriever) والبحث عن الاستعلام
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
query_text = "Which language should I use for machine learning coding?"
matched_docs = retriever.invoke(query_text)

print(f"\n🔍 الاستعلام: '{query_text}'")
print(f"📄 النتيجة المسترجعة: {matched_docs[0].page_content}")
print(f"🏷️ الميتا-داتا: {matched_docs[0].metadata}")

## Summary
Remote API embeddings provide zero-PyTorch execution, zero local disk overhead, and full LangChain compatibility.